In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
from tqdm.notebook import tqdm

sns.set(style="whitegrid")
train = pd.read_csv("/kaggle/input/hms-harmful-brain-activity-classification/train.csv")
test = pd.read_csv("/kaggle/input/hms-harmful-brain-activity-classification/test.csv")
all_df = pd.concat([train, test]).reset_index(drop=True)

display(train.head())
display(test.head())
display(all_df.head())



,eeg_id,eeg_sub_id,eeg_label_offset_seconds,spectrogram_id,spectrogram_sub_id,spectrogram_label_offset_seconds,label_id,patient_id,expert_consensus,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote
0,1628180742,0,0.0,353733,0,0.0,127492639,42516,Seizure,3,0,0,0,0,0
1,1628180742,1,6.0,353733,1,6.0,3887563113,42516,Seizure,3,0,0,0,0,0
2,1628180742,2,8.0,353733,2,8.0,1142670488,42516,Seizure,3,0,0,0,0,0
3,1628180742,3,18.0,353733,3,18.0,2718991173,42516,Seizure,3,0,0,0,0,0
4,1628180742,4,24.0,353733,4,24.0,3080632009,42516,Seizure,3,0,0,0,0,0


,spectrogram_id,eeg_id,patient_id
0,2207717,2578018731,34153
1,2207717,2578018731,34153
2,2207717,2578018731,34153
3,2207717,2578018731,34153
4,8352559,1706196508,37552


,eeg_id,eeg_sub_id,eeg_label_offset_seconds,spectrogram_id,spectrogram_sub_id,spectrogram_label_offset_seconds,label_id,patient_id,expert_consensus,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote
0,1628180742,0.0,0.0,353733,0.0,0.0,1.274926e+08,42516,Seizure,3.0,0.0,0.0,0.0,0.0,0.0
1,1628180742,1.0,6.0,353733,1.0,6.0,3.887563e+09,42516,Seizure,3.0,0.0,0.0,0.0,0.0,0.0
2,1628180742,2.0,8.0,353733,2.0,8.0,1.142670e+09,42516,Seizure,3.0,0.0,0.0,0.0,0.0,0.0
3,1628180742,3.0,18.0,353733,3.0,18.0,2.718991e+09,42516,Seizure,3.0,0.0,0.0,0.0,0.0,0.0
4,1628180742,4.0,24.0,353733,4.0,24.0,3.080632e+09,42516,Seizure,3.0,0.0,0.0,0.0,0.0,0.0


In [2]:
print(len(train.eeg_id.unique()))
print(len(train.spectrogram_id.unique()))
print(len(train.patient_id.unique()))
print(len(train.eeg_id.unique()) / len(train.patient_id.unique()))
print(len(train.spectrogram_id.unique()) / len(train.patient_id.unique()))



15396
10024
1910
8.060732984293194
5.2481675392670155


In [3]:
train_spectrogram_dir = (
    "/kaggle/input/hms-harmful-brain-activity-classification/train_spectrograms/"
)
train_spectrogram_files = os.listdir(train_spectrogram_dir)
print(f"There are {len(train_spectrogram_files)} train spectrogram parquets")
test_spectrogram_dir = (
    "/kaggle/input/hms-harmful-brain-activity-classification/test_spectrograms/"
)
test_spectrogram_files = os.listdir(test_spectrogram_dir)
print(f"There are {len(test_spectrogram_files)} test spectrogram parquets")
train_eeg_dir = "/kaggle/input/hms-harmful-brain-activity-classification/train_eegs/"
train_eeg_files = os.listdir(train_eeg_dir)
print(f"There are {len(train_eeg_files)} train eeg parquets")
test_eeg_dir = "/kaggle/input/hms-harmful-brain-activity-classification/test_eegs/"
test_eeg_files = os.listdir(test_eeg_dir)
print(f"There are {len(test_eeg_files)} test eeg parquets")




There are 10024 train spectrogram parquets
There are 1114 test spectrogram parquets
There are 15396 train eeg parquets
There are 1693 test eeg parquets


In [4]:
def get_files_info(files, file_dir):
    nan_ratio = []
    shapes = []
    for file in tqdm(files):
        data = np.array(pd.read_parquet(f"{file_dir}{file}"))
        nan_ratio.append(np.isnan(data).sum() / len(data.flatten()))
        shapes.append(data.shape)
    nan_ratio = np.array(nan_ratio)
    shapes = np.array(shapes)
    return nan_ratio, shapes




In [5]:
hypothesis0 = False
hypothesis1 = False

if len(train.spectrogram_id.unique()) / len(train.patient_id.unique()) < 5:
    hypothesis0 = True

if len(all_df.eeg_id.unique()) == (
    len(train.eeg_id.unique()) + len(test.eeg_id.unique())
):
    if len(all_df.patient_id.unique()) == (
        len(train.patient_id.unique()) + len(test.patient_id.unique())
    ):
        if len(all_df.spectrogram_id.unique()) == (
            len(train.spectrogram_id.unique()) + len(test.spectrogram_id.unique())
        ):
            hypothesis1 = True

print(f"hypothesis0: {hypothesis0}")
print(f"hypothesis1: {hypothesis1}")



hypothesis0: False
hypothesis1: False


In [6]:
hypotheses = []
hypotheses.append(len(test.eeg_id.unique()) == len(test))
hypotheses.append(len(test.spectrogram_id.unique()) == len(test))
hypotheses.append(len(test.patient_id.unique()) != len(test))
hypotheses.append(len(test_eeg_files) == len(test))
hypotheses.append(len(test_spectrogram_files) == len(test))
hypotheses.append(
    len(train.spectrogram_id.unique()) / len(train.patient_id.unique()) < 6
)
hypotheses.append(
    len(train.spectrogram_id.unique()) / len(train.patient_id.unique()) > 4
)

print(f"hypotheses: {hypotheses}")
hypotheses = all(hypotheses)
print(f"hyposetheses: {hypotheses}")



hypotheses: [False, False, True, False, False, True, True]
hyposetheses: False


In [7]:
vote_cols = [
    "seizure_vote",
    "lpd_vote",
    "gpd_vote",
    "lrda_vote",
    "grda_vote",
    "other_vote",
]

# Reduce smoothing to keep true vote ratios while avoiding zero‑division
SMOOTH_EPS = 1e-6

class_totals = train[vote_cols].sum()
overall_total = class_totals.sum()
global_mean = (class_totals / overall_total).to_dict()

patient_votes = train.groupby("patient_id")[vote_cols].sum() + SMOOTH_EPS
patient_totals = patient_votes.sum(axis=1).replace(0, 1e-6)
patient_probs = patient_votes.div(patient_totals, axis=0)
patient_probs = patient_probs.fillna(global_mean)  # fallback to global mean

eeg_votes = train.groupby("eeg_id")[vote_cols].sum() + SMOOTH_EPS
eeg_totals = eeg_votes.sum(axis=1).replace(0, 1e-6)
eeg_probs = eeg_votes.div(eeg_totals, axis=0)
eeg_probs = eeg_probs.fillna(global_mean)  # fallback to global mean

val = train.sample(frac=0.20, random_state=42)

val_true_tot = val[vote_cols].sum(axis=1).replace(0, np.nan)
val_true_prob = val[vote_cols].div(val_true_tot, axis=0)

candidate_alphas = np.linspace(0, 1, 21)  # 0.00, 0.05, ..., 1.00
best_alpha = None
best_kl = np.inf

for a in candidate_alphas:
    val_pred = pd.DataFrame(index=val.index)
    for col in vote_cols:
        val_pred[col] = val["patient_id"].map(patient_probs[col])
        val_pred[col] = val_pred[col].fillna(global_mean[col])
        val_pred[col] = a * val_pred[col] + (1 - a) * global_mean[col]
    row_sums = val_pred[vote_cols].sum(axis=1).replace(0, np.nan)
    val_pred[vote_cols] = val_pred[vote_cols].div(row_sums, axis=0)

    kl = np.sum(
        val_true_prob.values
        * np.log((val_true_prob.values + 1e-15) / (val_pred.values + 1e-15)),
        axis=1,
    )
    mean_kl = np.mean(kl)

    if mean_kl < best_kl:
        best_kl = mean_kl
        best_alpha = a

if best_alpha is not None:
    fine_candidates = np.arange(
        max(0, best_alpha - 0.05), min(1, best_alpha + 0.05) + 1e-8, 0.01
    )
    for a in fine_candidates:
        val_pred = pd.DataFrame(index=val.index)
        for col in vote_cols:
            val_pred[col] = val["patient_id"].map(patient_probs[col])
            val_pred[col] = val_pred[col].fillna(global_mean[col])
            val_pred[col] = a * val_pred[col] + (1 - a) * global_mean[col]
        row_sums = val_pred[vote_cols].sum(axis=1).replace(0, np.nan)
        val_pred[vote_cols] = val_pred[vote_cols].div(row_sums, axis=0)

        kl = np.sum(
            val_true_prob.values
            * np.log((val_true_prob.values + 1e-15) / (val_pred.values + 1e-15)),
            axis=1,
        )
        mean_kl = np.mean(kl)

        if mean_kl < best_kl:
            best_kl = mean_kl
            best_alpha = a

if best_alpha is None:
    best_alpha = 0.5
    print("No α selected during validation – using fallback α=0.5")

print(f"\nSelected α={best_alpha:.4f} with validation KL={best_kl:.5f}")

eeg_extra_candidates = [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
best_eeg_extra = 0.20  # default
best_combined_kl = best_kl

for ee_w in eeg_extra_candidates:
    patient_weight_adj = best_alpha * (1 - ee_w)
    global_weight = 1 - best_alpha
    eeg_weight = ee_w

    val_pred = pd.DataFrame(index=val.index)
    for col in vote_cols:
        val_pred[col] = val["patient_id"].map(patient_probs[col])
        val_pred[col] = val_pred[col].fillna(global_mean[col])
        val_pred[col] = (
            patient_weight_adj * val_pred[col] + global_weight * global_mean[col]
        )

        eeg_specific = val["eeg_id"].map(eeg_probs[col])
        eeg_specific = eeg_specific.fillna(global_mean[col])
        val_pred[col] = val_pred[col] + eeg_weight * (eeg_specific - val_pred[col])

    row_sums = val_pred[vote_cols].sum(axis=1).replace(0, np.nan)
    val_pred[vote_cols] = val_pred[vote_cols].div(row_sums, axis=0)

    kl = np.sum(
        val_true_prob.values
        * np.log((val_true_prob.values + 1e-15) / (val_pred.values + 1e-15)),
        axis=1,
    )
    mean_kl = np.mean(kl)

    if mean_kl < best_combined_kl:
        best_combined_kl = mean_kl
        best_eeg_extra = ee_w

print(
    f"Selected EEG extra weight={best_eeg_extra:.2f} with combined validation KL={best_combined_kl:.5f}"
)




Selected α=1.0000 with validation KL=0.43530
Selected EEG extra weight=0.30 with combined validation KL=0.22597


In [8]:
sub = pd.read_csv(
    "/kaggle/input/hms-harmful-brain-activity-classification/sample_submission.csv"
)
sub = sub.merge(test[["eeg_id", "patient_id"]], on="eeg_id", how="left")

patient_weight = best_alpha  # from validation
global_weight = 1 - best_alpha
eeg_extra_weight = best_eeg_extra  # tuned weight
patient_weight_adj = patient_weight * (1 - eeg_extra_weight)
eeg_weight = eeg_extra_weight

for col in vote_cols:
    sub[col] = sub["patient_id"].map(patient_probs[col])
    sub[col] = sub[col].fillna(global_mean[col])
    sub[col] = patient_weight_adj * sub[col] + global_weight * global_mean[col]

    eeg_specific = sub["eeg_id"].map(eeg_probs[col])
    eeg_specific = eeg_specific.fillna(global_mean[col])
    sub[col] = sub[col] + eeg_weight * (eeg_specific - sub[col])

row_sums = sub[vote_cols].sum(axis=1).replace(0, np.nan)
sub[vote_cols] = sub[vote_cols].div(row_sums, axis=0)

sub[vote_cols] = sub[vote_cols].clip(lower=1e-8)
row_sums = sub[vote_cols].sum(axis=1).replace(0, np.nan)
sub[vote_cols] = sub[vote_cols].div(row_sums, axis=0)

assert np.allclose(
    sub[vote_cols].sum(axis=1), 1.0, atol=1e-6
), "Row probabilities do not sum to 1."

sub = sub.drop(columns=["patient_id"])

sub.to_csv("/kaggle/working/submission.csv", index=False)
print("Submission written to /kaggle/working/submission.csv")

Submission written to /kaggle/working/submission.csv
